# Topic Analysis: 368 Topics Across Popularity Tiers

**Scope:** Topic-level analysis using BERTopic probabilities for all 368 topics. Statistical comparisons will use all three tiers (**Top / Middle / Trash**); pairwise contrasts (e.g., Top vs Trash) can be added as needed. Category-level statistics (taxonomy groups, Radway phases) are **excluded** here and will be handled in separate notebooks.

**Data sources (absolute paths):**
- Book features (wide): `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/book_features/book_taxonomy_main_props_wide.parquet`
- Book features (long): `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/book_features/book_taxonomy_main_props_long.parquet`
- Book topic probs: `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/topic_probabilities/book_topic_probs.parquet`
- Chapter topic probs: `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/topic_probabilities/chapter_topic_probs.parquet`
- Topic lookup: `/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage10_correlation_analysis/data_preparation/taxonomy_radway_eda/topic_lookup.parquet`

**Outputs:** `results/stage10_correlation_analysis/topic_analysis_all_368/` (figures, tables)


## Roadmap (Top / Middle / Trash)

1) Setup & paths
2) Load data + integrity checks (prob sums per book/segment, n_topics=368)
3) Rating-tier prep: map rating_class → {top, middle, trash}; use all three tiers; optional pairwise views (Top vs Trash) if needed
4) Helper utilities: plotting helper, Cliff's delta, prevalence metrics
5) Topic health metrics (prevalence, mass, concentration) — no taxonomy columns
6) Topic-level distributions & summaries (Top vs Middle vs Trash): medians, means, effect sizes, FDR-corrected tests
7) Leaderboards & exports (tables + optional plot stubs)
8) Optional: per-topic visualization hooks (violin/ECDF/box) for selected topic IDs

In [ ]:
# 1. Setup & imports
from __future__ import annotations

import os
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

# Set plotting defaults
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Inline plotting
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

import warnings
warnings.filterwarnings('ignore', message='.*IProgress not found.*')
warnings.filterwarnings('ignore', category=UserWarning, module='tqdm.auto')

PROJECT_ROOT = Path("/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor")
print(f"✓ PROJECT_ROOT: {PROJECT_ROOT}")

In [ ]:
# 2. Paths
DATA_PREP_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "data_preparation"
BOOK_FEATURES_DIR = DATA_PREP_DIR / "book_features"
TOPIC_PROBS_DIR = DATA_PREP_DIR / "topic_probabilities"
TAXONOMY_RADWAY_DIR = DATA_PREP_DIR / "taxonomy_radway_eda"

# Data file paths
BOOK_WIDE_PATH = BOOK_FEATURES_DIR / "book_taxonomy_main_props_wide.parquet"
BOOK_LONG_PATH = BOOK_FEATURES_DIR / "book_taxonomy_main_props_long.parquet"
BOOK_TOPIC_PROBS_PATH = TOPIC_PROBS_DIR / "book_topic_probs.parquet"
CHAPTER_TOPIC_PROBS_PATH = TOPIC_PROBS_DIR / "chapter_topic_probs.parquet"
TOPIC_LOOKUP_PATH = TAXONOMY_RADWAY_DIR / "topic_lookup.parquet"
GOODREADS_PATH = PROJECT_ROOT / "data" / "processed" / "goodreads.csv"

# Output directories
OUTPUT_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "topic_analysis_all_368"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
for d in [OUTPUT_DIR, FIG_DIR, TABLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Data paths:")
print(f"  Book features (wide): {BOOK_WIDE_PATH}")
print(f"  Book features (long): {BOOK_LONG_PATH}")
print(f"  Book topic probs: {BOOK_TOPIC_PROBS_PATH}")
print(f"  Chapter topic probs: {CHAPTER_TOPIC_PROBS_PATH}")
print(f"  Topic lookup: {TOPIC_LOOKUP_PATH}")
print(f"Outputs: {OUTPUT_DIR}")

In [ ]:
# 3. Load data
book_wide = pd.read_parquet(BOOK_WIDE_PATH)
book_long = pd.read_parquet(BOOK_LONG_PATH)
book_topic_probs = pd.read_parquet(BOOK_TOPIC_PROBS_PATH)
chapter_topic_probs = pd.read_parquet(CHAPTER_TOPIC_PROBS_PATH)
topic_lookup = pd.read_parquet(TOPIC_LOOKUP_PATH)

print("Shapes:")
print(f"  book_wide: {book_wide.shape}")
print(f"  book_long: {book_long.shape}")
print(f"  book_topic_probs: {book_topic_probs.shape}")
print(f"  chapter_topic_probs: {chapter_topic_probs.shape}")
print(f"  topic_lookup: {topic_lookup.shape}")

In [ ]:
# 4. Integrity checks
n_topics_expected = 368

# Check topics count
print("Unique topics (book):", book_topic_probs["topic_id"].nunique())
print("Unique topics (chapter):", chapter_topic_probs["topic_id"].nunique())

# Probability sums per book
book_sums = book_topic_probs.groupby("book_id")["prob"].sum()
print("Book prob sums — min/max:", book_sums.min(), book_sums.max())

# Probability sums per (book, chapter)
if {"book_id", "chapter_id"}.issubset(chapter_topic_probs.columns):
    chapter_sums = chapter_topic_probs.groupby(["book_id", "chapter_id"])["prob"].sum()
    print("Chapter prob sums — min/max:", chapter_sums.min(), chapter_sums.max())

# Basic cohort overlap
book_ids_probs = set(book_topic_probs["book_id"].unique())
book_ids_wide = set(book_wide["book_id"].unique())
print("Book IDs overlap (topic_probs ∩ wide):", len(book_ids_probs & book_ids_wide), "of", len(book_ids_probs))

# Rating class availability
if "rating_class" in book_wide.columns:
    print("Rating classes present:", book_wide["rating_class"].dropna().unique())
else:
    print("⚠️ rating_class column missing in book_wide")



In [ ]:
# 5. Rating-tier prep (Top / Middle / Trash)
# Expect rating_class in {"good","mid","bad"}; map to desired labels
rating_map = {
    "good": "top",
    "bad": "trash",
    "mid": "middle",
    "top": "top",
    "trash": "trash",
    "middle": "middle",
}

if "rating_class" not in book_wide.columns:
    raise KeyError("rating_class not found in book_wide; please add before analysis")

book_wide = book_wide.copy()
book_wide["rating_tier"] = book_wide["rating_class"].map(rating_map)

# Merge rating_tier onto topic probabilities
book_topic_probs = book_topic_probs.merge(
    book_wide[["book_id", "rating_tier"]], on="book_id", how="left"
)

# Use all three tiers by default
book_topic_probs_all = book_topic_probs.copy()

# Optional helper: Top vs Trash subset for pairwise contrasts if needed later
book_topic_probs_top_trash = book_topic_probs_all[book_topic_probs_all["rating_tier"].isin(["top", "trash"])].copy()

print("Books per tier (all):")
print(book_topic_probs_all.groupby("rating_tier")["book_id"].nunique())
print("\nBooks per tier (Top/Trash subset, optional):")
print(book_topic_probs_top_trash.groupby("rating_tier")["book_id"].nunique())



In [ ]:
# 6. Helper utilities

def show_plotly_fig(fig, save_html=True, output_dir=FIG_DIR):
    """Display Plotly figure; optionally save to HTML."""
    try:
        fig.show()
    except (ValueError, ImportError):
        pass
    if save_html and output_dir is not None:
        output_dir.mkdir(parents=True, exist_ok=True)
        html_path = output_dir / f"plot_{hash(str(fig.layout.title.text if fig.layout.title else 'figure'))}.html"
        fig.write_html(str(html_path))
        print(f"Saved interactive plot to: {html_path}")


def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """Cliff's delta effect size."""
    x = np.asarray(x)
    y = np.asarray(y)
    gt = np.sum(x[:, None] > y[None, :])
    lt = np.sum(x[:, None] < y[None, :])
    return (gt - lt) / (len(x) * len(y))


def compute_topic_health(df: pd.DataFrame, threshold: float = 0.001) -> pd.DataFrame:
    """Compute prevalence/mass/concentration per topic (no taxonomy columns)."""
    prevalence = (df["prob"] > threshold).groupby(df["topic_id"]).mean().rename("prevalence")
    mass = df.groupby("topic_id")["prob"].mean().rename("mass")
    # simple concentration proxy: max share / mean share per topic
    concentration = (
        df.groupby("topic_id")["prob"].max() / df.groupby("topic_id")["prob"].mean().replace(0, np.nan)
    ).rename("concentration_ratio")
    out = pd.concat([prevalence, mass, concentration], axis=1).reset_index()
    return out

